# Test Local Meeting Transcriber trên Colab

Notebook này chạy thử **Bước 1-3** đã bàn (cài backend, cài Ollama, chạy pipeline thật với 1 file audio) và có thêm phần tuỳ chọn để xem thử giao diện web qua trình duyệt (thay cho bản desktop Tauri, vì Tauri cần màn hình đồ hoạ mà Colab không có).

**Lưu ý quan trọng:**
- Colab là máy ảo **tạm thời** — đóng tab / hết phiên là mất hết, phải chạy lại từ đầu. Chỉ dùng để test nhanh, không phải nơi "để dành" project.
- Chọn Runtime → Change runtime type → GPU (T4) để Whisper chạy nhanh hơn (không bắt buộc, CPU vẫn chạy được).
- Chạy lần lượt từng ô theo thứ tự, đừng bỏ qua ô nào.

In [ ]:
# Kiểm tra có GPU không (chỉ để biết, không bắt buộc phải có)
!nvidia-smi || echo "Không có GPU — vẫn chạy được, chỉ chậm hơn."

## Bước 0 — Tải project lên Colab

Chạy ô dưới, một hộp thoại chọn file sẽ hiện ra — chọn file `local-meeting-transcriber.zip` mà Claude đã gửi.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # chọn local-meeting-transcriber.zip
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")

PROJECT_DIR = "/content/local-meeting-transcriber"
print("Đã giải nén vào:", PROJECT_DIR)
assert os.path.exists(f"{PROJECT_DIR}/backend/app/api/server.py"), "Không thấy đúng cấu trúc project — kiểm tra lại file zip đã upload."

## Bước 1 — Cài backend Python

Đây chính là "Bước 1" đã bàn — cài `requirements.txt` và các gói hệ thống cần thiết (ffmpeg cho xử lý audio).

In [ ]:
!apt -qq update && apt -qq install -y ffmpeg zstd > /dev/null
!pip install -q -r {PROJECT_DIR}/backend/requirements.txt
print("Cài xong.")

## Bước 2 — Cài và khởi động Ollama (để test bước tóm tắt biên bản)

Ollama là một server riêng — cài xong phải khởi động nền (`ollama serve`) trước khi dùng, giống hệt cách app desktop thật sẽ tự làm.

Dùng model `llama3.2:1b` (nhỏ, tải nhanh) để test cho lẹ — trên máy thật Thiên có thể dùng model lớn hơn (`llama3.2:3b`, `qwen2.5:7b`) cho chất lượng tốt hơn, xem `hardware.py` để biết máy nào nên dùng model nào.

In [ ]:
!apt -qq install -y zstd > /dev/null  # phòng khi chạy lại ô này mà bỏ qua Bước 1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

ollama_log = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
time.sleep(5)
print("Ollama đang chạy (PID:", ollama_proc.pid, ") — log ở /content/ollama.log")

In [ ]:
# Tải model — lần đầu mất vài phút tuỳ tốc độ mạng Colab
!ollama pull llama3.2:1b
!ollama list

## Bước 3 — Khởi động backend API (FastAPI)

Đây là API server thật mà app desktop sẽ gọi tới — chạy y hệt trên Colab.

In [ ]:
import subprocess, time, requests

backend_dir = f"{PROJECT_DIR}/backend"
backend_log = open("/content/uvicorn.log", "w")
backend_proc = subprocess.Popen(
    ["uvicorn", "app.api.server:app", "--host", "127.0.0.1", "--port", "8756"],
    cwd=backend_dir, stdout=backend_log, stderr=subprocess.STDOUT,
)
time.sleep(4)

res = requests.get("http://127.0.0.1:8756/api/hardware")
print("Cấu hình máy Colab phát hiện được:")
print(res.json())

Nếu ô trên lỗi (không kết nối được), chạy ô dưới để xem log lỗi backend:

In [ ]:
!tail -n 60 /content/uvicorn.log

## Test pipeline thật với 1 file audio

Đây là phần quan trọng nhất — xác nhận Whisper + Ollama chạy đúng từ đầu đến cuối. Upload 1 file ghi âm thật của Thiên (mp3/wav/m4a, nên để ngắn — 1-3 phút — để test nhanh trước).

In [ ]:
from google.colab import files

uploaded_audio = files.upload()  # chọn file audio thật
audio_filename = list(uploaded_audio.keys())[0]
print("Đã upload:", audio_filename)

In [ ]:
import requests, time

API_BASE = "http://127.0.0.1:8756"

with open(audio_filename, "rb") as f:
    res = requests.post(
        f"{API_BASE}/api/jobs",
        files={"file": (audio_filename, f)},
        data={"title": "Test trên Colab"},
    )
res.raise_for_status()
job_id = res.json()["job_id"]
print("Đã tạo job:", job_id)
print("Model được đề xuất:", res.json()["recommendation"])

In [ ]:
# Poll trạng thái job mỗi 3 giây (đơn giản hơn dùng SSE trong notebook)
while True:
    job = requests.get(f"{API_BASE}/api/jobs/{job_id}").json()
    print(f"[{job['stage']:>12}] {job['percent']:5.1f}%")
    if job["stage"] in ("done", "error"):
        break
    time.sleep(3)

if job["stage"] == "error":
    print("\nLỖI:", job["error_message"])
else:
    print("\n--- Biên bản ---")
    print(job["summary_markdown"])
    print("\n--- Vài đoạn transcript đầu ---")
    for seg in job["segments"][:5]:
        print(f"[{seg['start']:.1f}s] ({seg['speaker']}) {seg['text']}")

In [ ]:
# Nếu chạy thành công (stage == 'done'), tải file DOCX về máy để kiểm tra định dạng
if job["stage"] == "done":
    res = requests.post(f"{API_BASE}/api/jobs/{job_id}/export", params={"fmt": "docx"})
    out_path = "/content/bien_ban_test.docx"
    with open(out_path, "wb") as f:
        f.write(res.content)
    print("Đã lưu:", out_path)
    files.download(out_path)

## (Tuỳ chọn) Bước 4 — Xem thử giao diện web qua trình duyệt

Đây **không phải** bản desktop Tauri thật (Tauri cần màn hình đồ hoạ, Colab không có) — mà là chạy đúng file HTML/CSS/JS của `frontend/src/` (cùng logic, cùng giao diện) qua trình duyệt thường, để xem UI chạy đúng chưa trước khi đụng tới Rust/Tauri trên máy Thiên.

**Lưu ý:** thanh tiến trình dùng SSE (cập nhật real-time) có thể không chạy mượt qua đường proxy của Colab — nếu progress bar bị đứng, đó là do proxy chứ không phải lỗi code (đã test SSE chạy đúng khi gọi trực tiếp `127.0.0.1` ở các bước trên).

In [ ]:
from google.colab.output import eval_js

backend_url = eval_js("google.colab.kernel.proxyPort(8756)").rstrip("/")
print("Backend URL (qua Colab proxy):", backend_url)

In [ ]:
import shutil, http.server, socketserver, threading, functools

# Copy frontend ra 1 thư mục riêng rồi sửa API_BASE trỏ sang backend_url ở trên,
# thay vì sửa trực tiếp file gốc trong project.
COLAB_FRONTEND_DIR = "/content/colab_frontend"
shutil.rmtree(COLAB_FRONTEND_DIR, ignore_errors=True)
shutil.copytree(f"{PROJECT_DIR}/frontend/src", COLAB_FRONTEND_DIR)

main_js_path = f"{COLAB_FRONTEND_DIR}/main.js"
content = open(main_js_path, encoding="utf-8").read()
content = content.replace(
    'const API_BASE = "http://127.0.0.1:8756";',
    f'const API_BASE = "{backend_url}";',
)
open(main_js_path, "w", encoding="utf-8").write(content)

# Tắt server frontend cũ nếu ô này từng chạy trong phiên hiện tại.
try:
    httpd.shutdown()
    httpd.server_close()
except NameError:
    pass
except Exception:
    pass

# Dùng tham số `directory=` thay vì os.chdir() — os.chdir() đổi thư mục làm
# việc của CẢ tiến trình, còn SimpleHTTPRequestHandler đọc thư mục đó tại
# THỜI ĐIỂM xử lý từng request (không phải lúc khởi tạo), nên nếu code sau
# đó lỡ chdir đi chỗ khác (như trước đây), server sẽ phục vụ NHẦM thư mục.
# Dùng `directory=` thì server luôn phục vụ đúng thư mục đã chỉ định, không
# phụ thuộc thư mục làm việc hiện tại nữa.
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=COLAB_FRONTEND_DIR)

# Cũng để hệ điều hành tự chọn cổng trống (port=0) thay vì cố định 8080, vì
# Colab có thể còn giữ cổng cũ ở tầng hệ điều hành sau khi runtime tự restart.
httpd = socketserver.TCPServer(("", 0), handler)
FRONTEND_PORT = httpd.server_address[1]
threading.Thread(target=httpd.serve_forever, daemon=True).start()

print("Đang chạy frontend ở cổng nội bộ:", FRONTEND_PORT)
frontend_url = eval_js(f"google.colab.kernel.proxyPort({FRONTEND_PORT})")
print("\nMở link này để xem giao diện:")
print(frontend_url)

## Dọn dẹp (chạy khi test xong, không bắt buộc)

In [ ]:
backend_proc.terminate()
ollama_proc.terminate()
httpd.shutdown()
print("Đã tắt backend, Ollama, frontend server.")